# param-group-dict-list — worked example 1: differential-LR param groups

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `param-group-dict-list`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Passing parameters as a `list[dict]` lets different subsets get different hyperparameters. The classic case is a low learning rate on a pretrained encoder and a high one on a fresh head, built as two `{'params': ..., 'lr': ...}` dicts.

## Worked solution

We implement `make_param_groups(encoder, head, encoder_lr, head_lr)` returning a two-element list of dicts. Each dict materializes its `.parameters()` generator into a list (so re-iteration by downstream tooling is safe) and attaches the group's learning rate. We build a tiny encoder and head, construct the groups, hand them to a real `torch.optim.SGD`, and read back each group's `lr` from `optimizer.param_groups`. We print the two learning rates the optimizer stored to confirm the differential-LR setup was honored.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(0)

def make_param_groups(encoder, head, encoder_lr, head_lr):
    return [
        {'params': list(encoder.parameters()), 'lr': encoder_lr},
        {'params': list(head.parameters()), 'lr': head_lr},
    ]

encoder = nn.Linear(8, 8)
head = nn.Linear(8, 2)
groups = make_param_groups(encoder, head, 1e-4, 1e-2)
opt = t.optim.SGD(groups)
print('group lrs:', [g['lr'] for g in opt.param_groups])